# Unidad 3: Conectividad, Servicios Web y Consumo de APIs
**Materia:** Programación — Licenciatura en Negocios Digitales (3º año)
**Institución:** Universidad del Museo Social Argentino (UMSA)

---

## Introducción y Contexto de Negocio

En la economía de plataformas actual, **las aplicaciones no son sistemas aislados**. Los productos digitales exitosos se conectan de forma permanente con el ecosistema global: envían datos de ventas a un CRM (HubSpot), procesan cobros a través de pasarelas de pago (Mercado Pago, Stripe), disparan campañas de correo automáticas y recopilan métricas analíticas (Google Analytics API).

El puente técnico estándar que permite esta comunicación es la **API REST (Application Programming Interface)** sobre el protocolo web HTTP. En esta unidad aprenderemos los fundamentos del modelo Cliente-Servidor, manipularemos el formato JSON, integraremos APIs públicas/privadas y aprenderemos a gestionar claves y tokens de acceso de forma profesional y segura.

### Objetivos de Aprendizaje:
1. Comprender la arquitectura cliente-servidor y el protocolo HTTP (verbos, códigos de estado, headers).
2. Manipular y serializar datos en formato JSON en Python.
3. Consumir APIs RESTful públicas y privadas mediante solicitudes HTTP de la librería `requests`.
4. Estudiar casos prácticos de integración de negocios (Pasarelas de pago, CRM, analítica).
5. Gestionar credenciales y API Keys de manera segura sin exponerlas en el código.


## 1. Fundamentos de HTTP y APIs RESTful

El protocolo **HTTP** permite la transferencia de recursos en la web mediante el ciclo **Solicitud-Respuesta** (Request-Response).

### Verbos HTTP Clave:
- `GET`: Solicitar/recuperar información del servidor (sin alterar el estado).
- `POST`: Enviar datos al servidor para crear un nuevo recurso.
- `PUT`: Modificar o reemplazar un recurso existente en el servidor.
- `DELETE`: Eliminar un recurso del servidor.

### Códigos de Estado (Status Codes) Comunes:
- `200 OK`: La solicitud fue exitosa.
- `201 Created`: Recurso creado con éxito (típico de POST).
- `400 Bad Request`: El servidor no entendió la solicitud debido a datos inválidos en el cliente.
- `401 Unauthorized`: Falta autenticación o es inválida.
- `404 Not Found`: El recurso solicitado no existe.
- `500 Internal Server Error`: El servidor web falló al procesar la lógica interna.


## 2. Formato JSON y Consumo de APIs con `requests`

El formato **JSON (JavaScript Object Notation)** es la sintaxis universal de intercambio de datos. En Python, la librería `requests` facilita el envío de solicitudes y la traducción automática de payloads de JSON a diccionarios de Python.


In [ ]:
import requests
import json

# Consumiremos una API pública de pruebas llamada JSONPlaceholder
url_usuarios = "https://jsonplaceholder.typicode.com/users"

# Realizar una solicitud GET para traer la lista de usuarios
respuesta = requests.get(url_usuarios)

print(f"Estado de la respuesta: {respuesta.status_code}")

if respuesta.status_code == 200:
    # Convertimos la respuesta JSON en una lista de diccionarios de Python
    usuarios = respuesta.json()
    # Mostramos los primeros 3 usuarios
    print("\nPrimeros 3 usuarios traídos de la API:")
    for usr in usuarios[:3]:
        print(f"ID: {usr['id']} | Nombre: {usr['name']} | Compañía: {usr['company']['name']}")
else:
    print("Error al realizar la consulta.")


### Enviando datos mediante POST

Para registrar un nuevo recurso en un servidor de API, enviamos una solicitud `POST` junto con un payload en formato JSON y definimos los headers apropiados.


In [ ]:
url_creacion = "https://jsonplaceholder.typicode.com/posts"

# Datos del nuevo post que queremos crear en la base de datos de la API
datos_post = {
    "title": "Configuración Inicial de Stripe en Argentina",
    "body": "Paso a paso para integrar la API de Stripe usando cuentas internacionales.",
    "userId": 1
}

# Definimos headers (cabeceras) indicando que enviamos JSON
cabeceras = {
    "Content-Type": "application/json; charset=UTF-8"
}

# Enviamos la solicitud POST
respuesta_post = requests.post(url_creacion, data=json.dumps(datos_post), headers=cabeceras)

print(f"Status Code: {respuesta_post.status_code} (Esperado 201)")
print("Respuesta del servidor:")
print(json.dumps(respuesta_post.json(), indent=2, ensure_ascii=False))


## 3. Integración de Negocio y Seguridad de Credenciales

En entornos productivos, no debes incluir tus API Keys o Tokens secretos directamente escritos en el código fuente, ya que si subes tu repositorio a GitHub cualquiera podría robarlos. 

### Uso Seguro de Variables de Entorno en Entornos Locales y Google Colab
1. **Localmente**: Creamos un archivo `.env` que contenga `STRIPE_API_KEY=sk_test_12345` y lo cargamos con `python-dotenv`.
2. **Google Colab**: Usamos la sección de "Secrets" (icono de llave a la izquierda) y accedemos con `google.colab.userdata`.


In [ ]:
# Simulación del consumo seguro de API Keys usando colab secrets y un fallback de dotenv
import os

# Buscamos en Colab Secrets, si no existe usamos variables de entorno del sistema
try:
    from google.colab import userdata
    HUBSPOT_API_KEY = userdata.get('HUBSPOT_API_KEY')
    print("Clave de HubSpot cargada exitosamente desde Colab Secrets.")
except Exception:
    # Fallback para ejecución local
    HUBSPOT_API_KEY = os.getenv('HUBSPOT_API_KEY', 'default_mock_key_abc123')
    print("Clave cargada desde variables de entorno locales (fallback).")

print(f"Prefijo seguro de la Clave: {HUBSPOT_API_KEY[:5]}*****")


### Casos de Negocio: Sincronización Stripe -> CRM (HubSpot)

Veamos una simulación de cómo se estructura la lógica del backend para recibir un pago exitoso en Stripe y enviar el nuevo cliente a HubSpot CRM.


In [ ]:
# Simulación del SDK o la API REST de Stripe
class MockStripeAPI:
    @staticmethod
    def obtener_pago_reciente():
        # Devuelve un payload simulado de pago exitoso
        return {
            "id": "ch_3M4a9u2e",
            "monto": 150.00,
            "moneda": "USD",
            "estado": "succeeded",
            "cliente_email": "marta.gomez@gmail.com",
            "cliente_nombre": "Marta Gómez"
        }

# Simulación de la API REST de HubSpot CRM
class MockHubSpotCRM:
    def __init__(self, api_key):
        self.api_key = api_key
        
    def crear_o_actualizar_contacto(self, email, nombre):
        # En una API real haríamos: requests.post("https://api.hubapi.com/crm/v3/objects/contacts", ...)
        print(f"[HubSpot CRM API] Enviando solicitud con Token: {self.api_key[:4]}...")
        print(f"[HubSpot CRM API] Contacto Sincronizado -> Email: {email} | Nombre: {nombre}")
        return {"status": "success", "hubspot_id": "contact_991823"}

# Flujo de negocio integrado
pago = MockStripeAPI.obtener_pago_reciente()
if pago["estado"] == "succeeded":
    print(f"Pago de ${pago['monto']} {pago['moneda']} procesado en Stripe.")
    crm = MockHubSpotCRM(api_key=HUBSPOT_API_KEY)
    crm.crear_o_actualizar_contacto(email=pago["cliente_email"], nombre=pago["cliente_nombre"])


---

## Desafío Práctico (Trabajo Práctico 3)

**Consigna de Negocio (Integrador de Checkout y CRM):**
Debes automatizar la sincronización de leads de Mercado Pago a HubSpot.

1. Consume la API pública `https://jsonplaceholder.typicode.com/users` para simular que traes a los clientes que acaban de registrarse en tu tienda online.
2. Escribe una función llamada `sincronizar_clientes_tienda(api_key_crm: str)` que:
   - Recupere la lista de usuarios desde el endpoint anterior.
   - Filtre únicamente aquellos usuarios que pertenezcan a una empresa cuyo nombre contenga la palabra `"Group"` o `"LLC"` (clientes corporativos, más propensos a una venta B2B).
   - Simule el envío de estos contactos filtrados a la API de HubSpot imprimiendo en pantalla:
     `"[HubSpot Sync] Contacto corporativo creado -> [Nombre] ([Email]) | Empresa: [Compañía]"`
3. Llama a tu función pasando una API Key cargada de forma segura (puedes configurar una clave ficticia en Colab Secrets o usar un string de prueba predeterminado si ejecutas en entorno offline).

Implementa tu solución a continuación.


In [ ]:
# Escribe la resolución aquí
# 1. Función para consumir la API y sincronizar a HubSpot
# ...

# 2. Ejecutar y testear
# ...
